# Age Binning Strategies Analysis

This notebook explores different age binning strategies for fairness analysis using the `fairxai.experiments.age_binning` module.

**Goal**: Identify optimal age discretization strategy that:
- Maintains adequate sample sizes per group
- Balances group distributions
- Minimizes fairness sensitivity to binning choices

In [1]:
# Standard libraries
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Add project to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))

# Import FairXAI modules
from fairxai.experiments.age_binning import (
    create_binning_strategy,
    apply_binning,
    sensitive_attribute_distribution,
    compute_fairness_metrics,
    analyze_strategy_comprehensive,
    compare_strategies,
    compute_strategy_score
)
from fairxai.visualization.plots import plot_distribution

print(f"✓ Imports successful")
print(f"Project root: {project_root}")

✓ Imports successful
Project root: /home/miguel/Desktop/Dissertacao/Code/FairXAI


## 1. Load Raw Datasets

In [ ]:
# Load raw datasets
data_dir = project_root / 'data/raw/cardiac'

cleveland_df = pd.read_csv(data_dir / 'cleveland_standardized.csv')
kaggle_df = pd.read_csv(data_dir / 'kaggle_heart_standardized.csv')

# Verify required columns
required_cols = ['age_raw', 'sex', 'heart_disease']
for dataset_name, df in [('Cleveland', cleveland_df), ('Kaggle', kaggle_df)]:
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        print(f"⚠️  {dataset_name} missing columns: {missing}")
    else:
        print(f"✓ {dataset_name}: {df.shape[0]} samples, age range {df['age_raw'].min():.0f}-{df['age_raw'].max():.0f}")

datasets = {
    'cleveland': cleveland_df,
    'kaggle_heart': kaggle_df
}

NameError: name 'load_cleveland_data' is not defined

## 2. Test All Binning Strategies

In [ ]:
# Define strategies to test
strategies = ['fixed_10yr', 'fixed_5yr', 'clinical', 'quantile_3', 'quantile_5']

# Run comprehensive analysis for each dataset and strategy
all_results = []

for dataset_name, df in datasets.items():
    print(f"\n{'='*80}")
    print(f"Analyzing {dataset_name.upper()}")
    print(f"{'='*80}")
    
    for strategy_name in strategies:
        print(f"\n  Testing strategy: {strategy_name}...")
        
        try:
            result = analyze_strategy_comprehensive(
                df=df,
                strategy_name=strategy_name,
                dataset_name=dataset_name
            )
            
            if result:
                all_results.append(result)
                metrics = result['fairness_metrics']
                print(f"    ✓ Groups: {metrics['n_groups']}, "
                      f"Min size: {metrics['min_group_size']}, "
                      f"Balance CV: {metrics['group_balance_cv']:.3f}, "
                      f"SP diff: {metrics['max_sp_difference']:.3f}")
            else:
                print(f"    ✗ Strategy failed")
                
        except Exception as e:
            print(f"    ✗ Error: {e}")

print(f"\n✓ Completed {len(all_results)} analyses")

## 3. Compare Strategies

In [ ]:
# Create comparison dataframe
comparison_data = []
for key, result in results.items():
    dataset, strategy = key.split('_', 1)
    
    # Compute composite score (lower is better)
    # Penalize: min_group_size < 30, imbalance > 2.0, high CV
    min_size_penalty = max(0, 30 - result['min_group_size']) * 2
    imbalance_penalty = max(0, result['imbalance_ratio'] - 2.0) * 10
    cv_penalty = result['cv_group_sizes'] * 5
    
    composite_score = min_size_penalty + imbalance_penalty + cv_penalty
    
    comparison_data.append({
        'dataset': dataset,
        'strategy': strategy,
        'n_groups': result['n_groups'],
        'min_size': result['min_group_size'],
        'max_size': result['max_group_size'],
        'imbalance': result['imbalance_ratio'],
        'cv': result['cv_group_sizes'],
        'score': composite_score
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values(['dataset', 'score'])

print("Strategy Comparison (sorted by composite score, lower is better):")
print(comparison_df.to_string(index=False))

## 4. Top Recommendations per Dataset

In [ ]:
# Show top 3 strategies per dataset
print("TOP 3 STRATEGIES PER DATASET")
print("=" * 60)

for dataset in ['cleveland', 'kaggle']:
    print(f"\n{dataset.upper()} Dataset:")
    top3 = comparison_df[comparison_df['dataset'] == dataset].head(3)
    for idx, row in top3.iterrows():
        print(f"\n  {row['strategy']}:")
        print(f"    Score: {row['score']:.2f}")
        print(f"    Groups: {row['n_groups']}, Size range: {row['min_size']}-{row['max_size']}")
        print(f"    Imbalance: {row['imbalance']:.2f}, CV: {row['cv']:.3f}")

## 5. Visualize Distributions

In [ ]:
# Visualize group sizes for top strategies
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Group Size Distributions for Top Strategies', fontsize=16, y=1.02)

for dataset_idx, dataset in enumerate(['cleveland', 'kaggle']):
    top3 = comparison_df[comparison_df['dataset'] == dataset].head(3)
    
    for strat_idx, (_, row) in enumerate(top3.iterrows()):
        ax = axes[dataset_idx, strat_idx]
        key = f"{dataset}_{row['strategy']}"
        result = results[key]
        
        # Plot group sizes
        groups = list(result['group_sizes'].keys())
        sizes = list(result['group_sizes'].values())
        
        bars = ax.bar(range(len(groups)), sizes, color='steelblue', alpha=0.7)
        ax.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='Min threshold (30)')
        
        ax.set_title(f"{dataset.capitalize()}: {row['strategy']}\nScore: {row['score']:.1f}", fontsize=10)
        ax.set_xlabel('Age Group')
        ax.set_ylabel('Sample Count')
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels(groups, rotation=45, ha='right', fontsize=8)
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Fairness Sensitivity Analysis

In [ ]:
# Create heatmaps showing how metrics vary across strategies
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, dataset in enumerate(['cleveland', 'kaggle']):
    data = comparison_df[comparison_df['dataset'] == dataset].copy()
    
    # Create pivot table for heatmap
    metrics = ['min_size', 'imbalance', 'cv', 'score']
    heatmap_data = data[['strategy'] + metrics].set_index('strategy')
    
    # Normalize each metric to 0-1 scale for visualization
    normalized = heatmap_data.copy()
    for col in metrics:
        max_val = normalized[col].max()
        min_val = normalized[col].min()
        if max_val > min_val:
            normalized[col] = (normalized[col] - min_val) / (max_val - min_val)
    
    # Plot heatmap
    ax = axes[idx]
    sns.heatmap(normalized.T, annot=heatmap_data.T, fmt='.2f', 
                cmap='RdYlGn_r', ax=ax, cbar_kws={'label': 'Normalized Score'})
    ax.set_title(f'{dataset.capitalize()} Dataset: Metric Sensitivity', fontsize=12)
    ax.set_xlabel('Strategy')
    ax.set_ylabel('Metric')
    
plt.tight_layout()
plt.show()

## 7. Detailed Analysis: Best Strategy

In [ ]:
# Deep dive into best performing strategy for each dataset
print("DETAILED ANALYSIS OF BEST STRATEGIES")
print("=" * 60)

for dataset in ['cleveland', 'kaggle']:
    print(f"\n{dataset.upper()} Dataset")
    print("-" * 60)
    
    best = comparison_df[comparison_df['dataset'] == dataset].iloc[0]
    key = f"{dataset}_{best['strategy']}"
    result = results[key]
    
    print(f"\nBest Strategy: {best['strategy']}")
    print(f"Composite Score: {best['score']:.2f}")
    print(f"\nGroup Statistics:")
    print(f"  Number of groups: {result['n_groups']}")
    print(f"  Total samples: {result['total_samples']}")
    print(f"  Group size range: {result['min_group_size']} - {result['max_group_size']}")
    print(f"  Imbalance ratio: {result['imbalance_ratio']:.2f}")
    print(f"  Coefficient of Variation: {result['cv_group_sizes']:.3f}")
    
    print(f"\nGroup Size Breakdown:")
    for group, size in sorted(result['group_sizes'].items()):
        pct = (size / result['total_samples']) * 100
        print(f"  {group}: {size} ({pct:.1f}%)")
    
    print(f"\nInterpretation:")
    if result['min_group_size'] >= 30:
        print(f"  ✓ All groups meet minimum size requirement (≥30)")
    else:
        print(f"  ✗ Smallest group ({result['min_group_size']}) below threshold")
    
    if result['imbalance_ratio'] <= 2.0:
        print(f"  ✓ Good balance (imbalance ≤2.0)")
    else:
        print(f"  ⚠ Moderate imbalance ({result['imbalance_ratio']:.2f})")
    
    if result['cv_group_sizes'] <= 0.3:
        print(f"  ✓ Low variability in group sizes")
    else:
        print(f"  ⚠ Higher variability (CV={result['cv_group_sizes']:.3f})")

## Summary

### Key Findings

1. **Strategy Selection Matters**: Different binning strategies produce varying group sizes and balance characteristics that can significantly impact fairness metrics.

2. **Dataset-Specific Optimization**: The optimal strategy may differ between datasets based on sample size and age distribution.

3. **Trade-offs**: 
   - Fine-grained strategies (more groups) → better age resolution but smaller groups
   - Coarse strategies (fewer groups) → larger groups but less precision
   - Balance is key for robust fairness analysis

4. **Recommendations**:
   - **Cleveland Dataset**: Use strategy with best composite score (see above)
   - **Kaggle Dataset**: Use strategy with best composite score (see above)
   - Always verify minimum group sizes ≥30 for statistical validity
   - Monitor imbalance ratio to avoid biased fairness assessments

5. **Next Steps**:
   - Apply selected strategies to full preprocessing pipeline
   - Evaluate impact on model fairness metrics
   - Consider ensemble approach using multiple binning strategies for robustness